In [1]:
import os

# Create the scripting folder if it doesn't exist
os.makedirs("scripting", exist_ok=True)

In [3]:
os.getcwd()


'C:\\Users\\YYLB0843'

In [4]:
%%writefile scripting/script.py
import json
import sys
import os

def check_dict_diff(before: dict, after: dict):
    """Returns a set of keys that differ between two dictionaries."""
    before = before or {}
    after = after or {}
    all_keys = set(before.keys()).union(set(after.keys()))
    
    modified_keys = set()
    for key in all_keys:
        if before.get(key) != after.get(key):
            modified_keys.add(key)
    return modified_keys

def validate_tfplan(plan_file_path: str) -> bool:
    if not os.path.exists(plan_file_path):
        print(f"Error: File '{plan_file_path}' not found.")
        return False

    try:
        with open(plan_file_path, "r", encoding="utf-8") as f:
            plan_data = json.load(f)
    except Exception as e:
        print(f"Error reading/parsing JSON: {e}")
        return False

    resource_changes = plan_data.get("resource_changes", [])
    violations = []

    for rc in resource_changes:
        address = rc.get("address", "unknown_resource")
        change = rc.get("change", {})
        actions = change.get("actions", [])

        # 1. Ignore 'no-op' and 'read' operations (e.g., data sources)
        if actions == ["no-op"] or actions == ["read"]:
            continue

        # 2. Check for create action
        if actions == ["create"]:
            continue

        # 3. Check for delete / destroy actions
        if "delete" in actions:
            violations.append(
                f"[DENIED] Resource '{address}' has action '{','.join(actions)}'. Deletions are forbidden."
            )
            continue

        # 4. Check for update / modify actions
        if actions == ["update"]:
            before = change.get("before") or {}
            after = change.get("after") or {}

            changed_attrs = check_dict_diff(before, after)

            # Rule: Only 'tags' (or 'tags_all') can be modified
            disallowed_attrs = [attr for attr in changed_attrs if attr not in ("tags", "tags_all")]
            if disallowed_attrs:
                violations.append(
                    f"[DENIED] Resource '{address}' modifies non-tag attributes: {disallowed_attrs}."
                )
                continue

            # If tags were modified, verify that ONLY 'GitCommitHash' changed
            for tag_attr in ("tags", "tags_all"):
                if tag_attr in changed_attrs:
                    before_tags = before.get(tag_attr) or {}
                    after_tags = after.get(tag_attr) or {}
                    changed_tag_keys = check_dict_diff(before_tags, after_tags)

                    disallowed_tags = [k for k in changed_tag_keys if k != "GitCommitHash"]
                    if disallowed_tags:
                        violations.append(
                            f"[DENIED] Resource '{address}' modifies unauthorized tag keys: {disallowed_tags}. Only 'GitCommitHash' is allowed."
                        )
            continue

        # Any other action pattern (e.g. replace -> ["delete", "create"])
        violations.append(
            f"[DENIED] Resource '{address}' requires an unsupported action flow: {actions}."
        )

    # Print summary & action decision
    print("=" * 60)
    print(f"Plan Evaluation: {os.path.basename(plan_file_path)}")
    print("=" * 60)

    if violations:
        print("\nACTION REQUIRED: DO NOT PROCEED WITH TERRAFORM APPLY\n")
        print("Violations:")
        for violation in violations:
            print(f"  - {violation}")
        print("\nResult: Apply Rejected ❌\n")
        return False
    else:
        print("\nACTION REQUIRED: PROCEED WITH TERRAFORM APPLY\n")
        print("Result: Apply Approved ✅\n")
        return True

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python script.py <path_to_tfplan.json>")
        sys.exit(1)

    plan_path = sys.argv[1]
    is_valid = validate_tfplan(plan_path)
    sys.exit(0 if is_valid else 1)

Writing scripting/script.py


In [5]:
# Run against a specific plan file
!python scripting/script.py scripting/tfplan-1.json

Plan Evaluation: tfplan-1.json

ACTION REQUIRED: PROCEED WITH TERRAFORM APPLY



Traceback (most recent call last):
  File "C:\Users\YYLB0843\scripting\script.py", line 109, in <module>
    is_valid = validate_tfplan(plan_path)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\YYLB0843\scripting\script.py", line 100, in validate_tfplan
    print("Result: Apply Approved \u2705\n")
  File "C:\Users\YYLB0843\AppData\Local\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 23: character maps to <undefined>


In [6]:
%%writefile scripting/script.py
import json
import sys
import os

def check_dict_diff(before: dict, after: dict):
    """Returns a set of keys that differ between two dictionaries."""
    before = before or {}
    after = after or {}
    all_keys = set(before.keys()).union(set(after.keys()))
    
    modified_keys = set()
    for key in all_keys:
        if before.get(key) != after.get(key):
            modified_keys.add(key)
    return modified_keys

def validate_tfplan(plan_file_path: str) -> bool:
    if not os.path.exists(plan_file_path):
        print(f"Error: File '{plan_file_path}' not found.")
        return False

    try:
        with open(plan_file_path, "r", encoding="utf-8") as f:
            plan_data = json.load(f)
    except Exception as e:
        print(f"Error reading/parsing JSON: {e}")
        return False

    resource_changes = plan_data.get("resource_changes", [])
    violations = []

    for rc in resource_changes:
        address = rc.get("address", "unknown_resource")
        change = rc.get("change", {})
        actions = change.get("actions", [])

        # 1. Ignore 'no-op' and 'read' operations (e.g., data sources)
        if actions == ["no-op"] or actions == ["read"]:
            continue

        # 2. Check for create action
        if actions == ["create"]:
            continue

        # 3. Check for delete / destroy actions
        if "delete" in actions:
            violations.append(
                f"[DENIED] Resource '{address}' has action '{','.join(actions)}'. Deletions are forbidden."
            )
            continue

        # 4. Check for update / modify actions
        if actions == ["update"]:
            before = change.get("before") or {}
            after = change.get("after") or {}

            changed_attrs = check_dict_diff(before, after)

            # Rule: Only 'tags' (or 'tags_all') can be modified
            disallowed_attrs = [attr for attr in changed_attrs if attr not in ("tags", "tags_all")]
            if disallowed_attrs:
                violations.append(
                    f"[DENIED] Resource '{address}' modifies non-tag attributes: {disallowed_attrs}."
                )
                continue

            # If tags were modified, verify that ONLY 'GitCommitHash' changed
            for tag_attr in ("tags", "tags_all"):
                if tag_attr in changed_attrs:
                    before_tags = before.get(tag_attr) or {}
                    after_tags = after.get(tag_attr) or {}
                    changed_tag_keys = check_dict_diff(before_tags, after_tags)

                    disallowed_tags = [k for k in changed_tag_keys if k != "GitCommitHash"]
                    if disallowed_tags:
                        violations.append(
                            f"[DENIED] Resource '{address}' modifies unauthorized tag keys: {disallowed_tags}. Only 'GitCommitHash' is allowed."
                        )
            continue

        # Any other action pattern (e.g. replace -> ["delete", "create"])
        violations.append(
            f"[DENIED] Resource '{address}' requires an unsupported action flow: {actions}."
        )

    # Print summary & action decision
    print("=" * 60)
    print(f"Plan Evaluation: {os.path.basename(plan_file_path)}")
    print("=" * 60)

    if violations:
        print("\nACTION REQUIRED: DO NOT PROCEED WITH TERRAFORM APPLY\n")
        print("Violations:")
        for violation in violations:
            print(f"  - {violation}")
        print("\nResult: Apply Rejected [REJECTED]\n")
        return False
    else:
        print("\nACTION REQUIRED: PROCEED WITH TERRAFORM APPLY\n")
        print("Result: Apply Approved [APPROVED]\n")
        return True

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python script.py <path_to_tfplan.json>")
        sys.exit(1)

    plan_path = sys.argv[1]
    is_valid = validate_tfplan(plan_path)
    sys.exit(0 if is_valid else 1)

Overwriting scripting/script.py


In [7]:
!python scripting/script.py scripting/tfplan-1.json

Plan Evaluation: tfplan-1.json

ACTION REQUIRED: PROCEED WITH TERRAFORM APPLY

Result: Apply Approved [APPROVED]



In [8]:
!python scripting/script.py scripting/tfplan-2.json

Plan Evaluation: tfplan-2.json

ACTION REQUIRED: PROCEED WITH TERRAFORM APPLY

Result: Apply Approved [APPROVED]



In [9]:
!python scripting/script.py scripting/tfplan-3.json

Plan Evaluation: tfplan-3.json

ACTION REQUIRED: DO NOT PROCEED WITH TERRAFORM APPLY

Violations:
  - [DENIED] Resource 'module.aihub.azapi_resource.hub' modifies non-tag attributes: ['output'].
  - [DENIED] Resource 'module.aihub.azapi_resource.project' modifies non-tag attributes: ['body', 'output'].

Result: Apply Rejected [REJECTED]

